# Lab 10.3 &mdash; Read the Trace

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 5 &middot; Module 10 &mdash; Ethics &amp; Responsible AI in Agentic Systems**

### What you'll do
- Read a ReAct trace as (tool, arg, observation) steps
- Find the step where a wrong/unknown tool was called
- Catch an ungrounded argument the final answer hides

> **How this lab works (near-real):** read the **Concept**, fill the real `___` blanks in **Build it**, then **run it &amp; observe**. The responsible-AI logic here &mdash; injection defence, least privilege, trace-reading, fairness, the eval loop, the guardrails &mdash; is **real, deterministic Python** you can run offline. The **Advanced agent labs (10&ndash;12)** additionally drive a **real Groq model** through `create_agent`: **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a responsible agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`). The agent labs use a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`); the guardrail/eval/trace logic is genuine rule-based Python. If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. This is the **course finale** &mdash; Lab 5.2: responsible-AI frameworks &amp; **debugging agent errors**.

**Reference:** [Module 10 slides &mdash; Read the trace — a broken run](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 10 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-10-03")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-5 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=5):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The trace is your **#1 debugging surface** (deck slide 14). The final answer often looks plausible; the
trace shows **where and why** a run went wrong &mdash; a **wrong tool** at one step, an **ungrounded
argument** at another. You debug an agent by reading its reasoning like a transcript. Here you localise two
classic bugs from a **captured run** (a real incident, recorded as steps so the diagnosis is deterministic
&mdash; in Lab 10 you read the same shape off a live agent's real messages).

In [ ]:
# A captured broken run, recorded as steps of (tool, arg, observation).
TRACE = [
    ("lookup_order", "revenue", "unknown tool: lookup_order"),   # wrong tool
    ("compute", "0.15 * 100", 15.0),                             # 100 was never grounded!
]
GROUNDED = {"120"}   # the only value actually retrieved from the report was 120
print("steps:", len(TRACE), "| grounded values:", GROUNDED)

## Build it
Complete `find_wrong_tool`, `used_tools`, and `find_ungrounded` &mdash; the three reads that localise a
bug from a trace.

In [ ]:
def find_wrong_tool(steps):
    # index of the first step whose observation reports an unknown tool, else -1
    for i, (tool, arg, obs) in enumerate(steps):
        if ___:   # TODO: the observation string mentions an unknown tool
            return i
    return -1

def used_tools(steps):
    return ___   # TODO: the tool name of each step, in order

def find_ungrounded(steps, grounded):
    # a compute step whose argument uses a number that was never grounded
    for i, (tool, arg, obs) in enumerate(steps):
        if tool == "compute" and ___:   # TODO: True when NONE of the grounded values appears in str(arg)
            return i
    return -1

try:
    print("wrong tool at step:", find_wrong_tool(TRACE))
    print("tools used        :", used_tools(TRACE))
    print("ungrounded at step:", find_ungrounded(TRACE, GROUNDED))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- `find_wrong_tool` localises the bug to **step 0** &mdash; the model called a tool it wasn't given.
- `find_ungrounded` catches **step 1**: the compute used `100`, a number never retrieved (only `120` was grounded).
- The final answer alone would hide both; the **trace** is what makes an agent debuggable. Same reads work on real messages (Lab 10).

## Your turn (open task &mdash; no grader)
Add a third step to `TRACE` &mdash; e.g. a second compute that *does* use a grounded value &mdash; and re-run.
**What good looks like:** `find_ungrounded` flags only the step that invented a number, and skips the grounded
one. That's the exact check you'll run against a live agent's trace in Lab 10.

---
### Key takeaway
The trace shows not just THAT a run failed but WHERE and WHY -- a wrong tool at step 1, an ungrounded number at step 2. The final answer alone hides both. Transparency and debuggability are the same thing.

[&#8592; All Module 10 labs](./index.html) &nbsp;&middot;&nbsp; [Module 10 slides](../../presentation/day5-module10-ethics-responsible-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>